# TP Jour 1 — Choix des Datasets Kaggle (Mono ou Multi-Sources), HDFS & Couche Bronze

Bienvenue dans ce fil rouge **Data Lakehouse** ! Vous êtes libres de choisir votre thématique métier et de combiner **un dataset multi-tables ou plusieurs datasets publics complémentaires sur Kaggle** afin de les croiser par jointure.

### Objectifs de la séance
1. **Cadrer votre sujet métier** et sélectionner **un ou plusieurs datasets Kaggle** (~50 à 500 Mo au total) partageant au moins un axe de jointure (date/heure, zone géographique, code ISO/catégorie, identifiant métier).
2. **Explorer le cluster HDFS réel** (`1 NameNode` + `3 DataNodes`) configuré avec `dfs.blocksize = 16 Mo` et `dfs.replication = 2`.
3. **Ingérer vos fichiers bruts multi-sources sur HDFS** (`/lakehouse/bronze/raw/<source>/...`) et observer physiquement leur découpage en blocs de 16 Mo.
4. **Réaliser un crash-test en direct** : couper `datanode2` et observer la détection de panne + l'auto-réparation vers le 3e DataNode en ~50 secondes.
5. **Construire votre couche Bronze** (`/lakehouse/bronze/`) au format **Delta Lake** avec traçabilité de la source (`_source_dataset`, `_source_file`, `_ingested_at`) et inspecter vos jobs sur **Spark Web UI** (`http://localhost:4040`).

---
### Fiche de cadrage de votre projet *(À compléter par le groupe)*
- **Titre du sujet :** `Analyse Big Data des retards de livraison Olist`
- **Dataset(s) Kaggle retenu(s) (`auteur/nom-du-dataset`) :**
  1. Source principale : `olistbr/brazilian-ecommerce`
  2. Source(s) complémentaire(s) pour jointure : `orders`, `customers`, `order_items`, `products`, `payments`, `reviews` (tables du même dataset)
- **Clé(s) de jointure envisagée(s) entre sources / tables :** `order_id`, `customer_id`, `product_id` *(ex. date + ville, code aéroport, code postal, catégorie...)*
- **Problématique métier principale :** Comprendre et prédire les retards de livraison des commandes Olist.
- **Variables clés retenues (temporelles, quantitatives, catégorielles) :** dates de commande/livraison, prix, frais de port, note, état client, catégorie produit.

## 1. Téléchargement de vos Datasets Kaggle dans `/workspace/data_raw/<source>/`

Renseignez dans la liste `KAGGLE_DATASETS` un ou plusieurs identifiants Kaggle (ex. `["auteur1/dataset-principal", "auteur2/meteo-ou-referentiel"]`), ou déposez directement vos fichiers/sous-dossiers dans `data_raw/`.
- Chaque dataset est isolé dans son propre sous-dossier `data_raw/<nom-dataset>/` pour éviter toute collision de noms (`train.csv`, `metadata.csv`, etc.) entre plusieurs sources.
- **Critère important :** Vérifiez qu'au moins un fichier dépasse **20 à 32 Mo** afin d'observer sa division en plusieurs blocs HDFS de **16 Mo** !

In [ ]:
import re
import shutil
from pathlib import Path

DATA_RAW = Path("/workspace/data_raw")
DATA_RAW.mkdir(parents=True, exist_ok=True)
SUPPORTED_EXTS = {".csv", ".parquet", ".json", ".jsonl", ".tsv"}

# TODO : Renseignez un ou plusieurs datasets Kaggle à combiner (format "auteur/nom-du-dataset")
KAGGLE_DATASETS: list[str] = ["olistbr/brazilian-ecommerce"]

for slug in KAGGLE_DATASETS:
    slug = slug.strip()
    if not slug:
        continue
    subdir = re.sub(r"[^a-zA-Z0-9._-]+", "_", slug.split("/")[-1]).strip("_") or "dataset"
    target_dir = DATA_RAW / subdir
    existing = [f for f in target_dir.rglob("*") if f.is_file() and f.suffix.lower() in SUPPORTED_EXTS] if target_dir.exists() else []
    if not existing:
        import kagglehub
        print(f"Téléchargement de {slug} -> {target_dir}/ ...")
        cache_dir = Path(kagglehub.dataset_download(slug))
        target_dir.mkdir(parents=True, exist_ok=True)
        for src_file in sorted(cache_dir.rglob("*")):
            if src_file.is_file() and src_file.suffix.lower() in SUPPORTED_EXTS:
                dest_file = target_dir / src_file.relative_to(cache_dir)
                dest_file.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(src_file, dest_file)

# On travaille uniquement avec la copie du dataset sélectionné pour éviter les doublons.
local_files = sorted(
    f for f in (DATA_RAW / "brazilian-ecommerce").rglob("*")
    if f.is_file() and f.suffix.lower() in SUPPORTED_EXTS
)

assert len(local_files) > 0, (
    "Aucun fichier détecté dans /workspace/data_raw/. "
    "Renseignez `KAGGLE_DATASETS` ci-dessus ou utilisez `task data:download DATASET='auteur1/ds1,auteur2/ds2'`."
)

print(f"Inventaire local multi-sources dans {DATA_RAW} ({len(local_files)} fichier(s)) :")
for f in local_files:
    rel = f.relative_to(DATA_RAW)
    size_mb = f.stat().st_size / (1024 * 1024)
    expected_blocks = max(1, int(-(-f.stat().st_size // (16 * 1024 * 1024))))
    print(f" - {str(rel):<55} {size_mb:>7.2f} Mo  (~{expected_blocks} bloc(s) HDFS de 16 Mo)")

## 2. Démarrage de la `SparkSession` pré-configurée sur HDFS

Grâce au fichier `config/spark-defaults.conf`, Spark pointe automatiquement vers `hdfs://namenode:9000` avec **Delta Lake** activé.

### Exercice 2.1 — Initialiser `SparkSession` et contrôler la configuration HDFS
1. Créez une session Spark nommée `"J1_HDFS_Ingestion_Bronze"`.
2. Récupérez via `spark.conf.get(...)` la valeur de `"spark.hadoop.fs.defaultFS"`, `"spark.hadoop.dfs.blocksize"` et `"spark.hadoop.dfs.replication"`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("J1_HDFS_Ingestion_Bronze").getOrCreate()

# Lecture des paramètres réellement appliqués par Hadoop (avec repli sur spark.conf).
hconf = spark.sparkContext._jsc.hadoopConfiguration()
def hdfs_config(key, default=None):
    return hconf.get(key) or spark.conf.get("spark.hadoop." + key, default)

default_fs = hdfs_config("fs.defaultFS")
block_size = hdfs_config("dfs.blocksize")
replication = hdfs_config("dfs.replication")

print("Spark version       :", spark.version)
print("Default FileSystem  :", default_fs)
print("HDFS Block Size (o) :", block_size)
print("HDFS Replication    :", replication)
assert default_fs and default_fs.startswith("hdfs://"), "HDFS non configuré : vérifiez spark-defaults.conf."


## 3. Copie des fichiers bruts vers HDFS & Inspection des Blocs de 16 Mo

La cellule suivante téléverse vos fichiers bruts de `/workspace/data_raw` vers `/lakehouse/bronze/raw/` sur HDFS, puis interroge le NameNode pour afficher la localisation exacte de chaque bloc de 16 Mo sur les 3 DataNodes (`datanode1`, `datanode2`, `datanode3`).

In [ ]:
jvm = spark._jvm
hadoop_conf = spark._jsc.hadoopConfiguration()
fs = jvm.org.apache.hadoop.fs.FileSystem.get(hadoop_conf)
PathHDFS = jvm.org.apache.hadoop.fs.Path

raw_hdfs_dir = PathHDFS("/lakehouse/bronze/raw")
fs.mkdirs(raw_hdfs_dir)

for file_path in local_files:
    rel_posix = file_path.relative_to(DATA_RAW).as_posix()
    src = PathHDFS(f"file://{file_path}")
    dst = PathHDFS(f"/lakehouse/bronze/raw/{rel_posix}")
    fs.mkdirs(dst.getParent())
    fs.copyFromLocalFile(False, True, src, dst)
    print(f"Téléversé vers HDFS : {dst}")

In [ ]:
# Inspection récursive du découpage en blocs de 16 Mo sur l'ensemble des sources ingérées
remote_iter = fs.listFiles(raw_hdfs_dir, True)
statuses = []
while remote_iter.hasNext():
    statuses.append(remote_iter.next())
statuses.sort(key=lambda s: s.getLen(), reverse=True)
assert statuses, "Aucun fichier ingéré dans HDFS."
largest_hdfs_path = statuses[0].getPath().toString()

for status in statuses:
    path = status.getPath()
    rel_hdfs = path.toString().split("/lakehouse/bronze/raw/", 1)[-1]
    length_mb = status.getLen() / (1024 * 1024)
    blocks = fs.getFileBlockLocations(status, 0, status.getLen())
    print(f"Fichier : {rel_hdfs} ({length_mb:.2f} Mo | repl={status.getReplication()})")
    for idx, blk in enumerate(blocks):
        print(f"   Bloc #{idx} ({blk.getLength() / (1024*1024):5.2f} Mo) -> DataNodes : {list(blk.getHosts())}")

### Questions d'analyse (à vérifier aussi sur http://localhost:9870)
1. Comparez sur l'interface Web du NameNode (`Utilities > Browse the file system`) le nombre de blocs de votre fichier le plus volumineux avec sa taille divisée par 16 Mo.
2. Un fichier de 3 Mo bloque-t-il réellement 16 Mo d'espace disque sur les DataNodes ? Pourquoi ?
3. Pourquoi avoir configuré `dfs.replication = 2` sur un cluster à `3` DataNodes ?

---
## 4. Crash-Test en Direct : Panne d'un DataNode et Auto-Réparation (~50s)

Dans `config/hdfs-site.xml`, le délai de détection d'un DataNode mort a été abaissé de 10m30s à **~50 secondes** :
$$2 \times \text{recheck-interval (10s)} + 10 \times \text{heartbeat.interval (3s)} = 50\text{ s}$$

### Protocole à réaliser depuis votre terminal :
1. Inspectez l'état initial des blocs : `task hdfs:fsck`
2. Arrêtez brutalement le DataNode 2 :
   ```bash
   task hdfs:crash-dn2   # équivalent à : docker stop datanode2
   ```
3. Observez l'onglet **Datanodes** sur `http://localhost:9870` pendant ~50s : dès que `datanode2` passe à **Dead**, le NameNode détecte les blocs sous-répliqués et ordonne leur copie automatique vers le 3e DataNode survivant !

### Exercice 4.1 — Vérifier la continuité de service et la *Data Locality*
Pendant que `datanode2` est éteint, lisez votre plus gros fichier (`largest_hdfs_path`) avec PySpark, affichez `df_check.count()` ainsi que `df_check.rdd.getNumPartitions()`, et expliquez le lien avec le nombre de blocs HDFS.

*(Pensez ensuite à rallumer le nœud avec `task hdfs:restore-dn2` !)*

In [ ]:
print("Fichier cible du test de résilience :", largest_hdfs_path)

path_lower = largest_hdfs_path.lower()
if path_lower.endswith((".csv", ".tsv")):
    reader = spark.read.option("header", "true").option("inferSchema", "true")
    if path_lower.endswith(".tsv"):
        reader = reader.option("sep", "\t")
    df_check = reader.csv(largest_hdfs_path)
elif path_lower.endswith(".parquet"):
    df_check = spark.read.parquet(largest_hdfs_path)
elif path_lower.endswith((".json", ".jsonl")):
    df_check = spark.read.json(largest_hdfs_path)
else:
    raise ValueError(f"Format non supporté : {largest_hdfs_path}")

print(f"Nombre total de lignes lues       : {df_check.count():,}")
print(f"Nombre de partitions Spark créées : {df_check.rdd.getNumPartitions()}")
print("Note : le nombre de partitions dépend du découpage des fichiers et de la configuration Spark.")


---
## 5. Construction de votre Couche Bronze en Delta Lake (`/lakehouse/bronze/`)

### Exercice 5.1 — Convertir vos fichiers bruts (mono ou multi-datasets) en tables Delta Lake Bronze
Dans une architecture **Médaillon multi-sources**, la couche **Bronze** conserve chaque table brute fidèle à sa source tout en l'enrichissant de métadonnées de lignage (*lineage*) indispensables avant les jointures du Jour 2 :
- `_ingested_at` : `F.current_timestamp()`
- `_source_dataset` : `F.lit(dataset_name)` *(identifie le dataset d'origine)*
- `_source_file` : `F.col("_metadata.file_path")`

> **Attention aux noms de colonnes Delta/Parquet :** Si vos fichiers CSV bruts contiennent des espaces ou caractères spéciaux dans les en-têtes, normalisez-les simplement en `snake_case` (ex. `.toDF(*[c.strip().lower().replace(" ", "_") for c in df.columns])`).

Complétez la cellule ci-dessous pour ingérer chacune des tables utiles de vos datasets dans `/lakehouse/bronze/<nom_table>`.

In [ ]:
# Tables Olist : une table Bronze par CSV, sans multiplication des sources.
OLIST_FILES = {
    "orders": "olist_orders_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}
BRONZE_SOURCES: dict[str, tuple[str, str]] = {
    name: ("olistbr/brazilian-ecommerce", f"/lakehouse/bronze/raw/brazilian-ecommerce/{filename}")
    for name, filename in OLIST_FILES.items()
}

assert BRONZE_SOURCES, "Aucune table Bronze définie."
for table_name, (dataset_name, raw_path) in BRONZE_SOURCES.items():
    bronze_delta_path = f"/lakehouse/bronze/{table_name}"
    assert fs.exists(PathHDFS(raw_path)), f"CSV absent sur HDFS : {raw_path}"

    # Le CSV des avis contient des commentaires sur plusieurs lignes : multiLine est indispensable.
    # Le fichier est lu en une seule source, sans concaténation des copies locales.
    df_raw = (
        spark.read.format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .option("encoding", "UTF-8")
        .option("multiLine", "true" if table_name == "reviews" else "false")
        .option("quote", '"')
        .option("escape", '"')
        .option("mode", "PERMISSIVE")
        .load(raw_path)
    )
    normalized = [re.sub(r"[^a-zA-Z0-9_]", "_", c.strip().lower()) for c in df_raw.columns]
    assert len(normalized) == len(set(normalized)), f"Noms de colonnes dupliqués dans {table_name}"
    df_raw = df_raw.toDF(*normalized)

    # Les métadonnées de traçabilité restent constantes pour chaque fichier ingéré.
    df_bronze = (
        df_raw.withColumn("_ingested_at", F.current_timestamp())
        .withColumn("_source_dataset", F.lit(dataset_name))
        .withColumn("_source_file", F.lit(raw_path))
    )
    (df_bronze.write.format("delta")
     .mode("overwrite")
     .option("overwriteSchema", "true")
     .save(bronze_delta_path))

    cnt = spark.read.format("delta").load(bronze_delta_path).count()
    print(f"[BRONZE OK] {bronze_delta_path:<38} (source={dataset_name:<28}) -> {cnt:>10,} lignes")

# Détection des notes non numériques pour préparer J2.
reviews_check = spark.read.format("delta").load("/lakehouse/bronze/reviews")
reviews_check.select(
    F.count("*").alias("reviews_total"),
    F.sum(F.expr("CASE WHEN review_score IS NOT NULL AND try_cast(review_score AS DOUBLE) IS NULL THEN 1 ELSE 0 END")).alias("scores_invalides")
).show(truncate=False)


In [ ]:
# Auto-contrôle de fin de Jour 1 : au moins une table Delta valide dans /lakehouse/bronze/
bronze_root = PathHDFS("/lakehouse/bronze")
bronze_delta_tables = [
    s.getPath().getName()
    for s in fs.listStatus(bronze_root)
    if s.isDirectory() and fs.exists(PathHDFS(s.getPath(), "_delta_log"))
]
assert len(bronze_delta_tables) > 0, "Aucune table Delta détectée dans /lakehouse/bronze/ !"
print("Tables Delta Bronze validées :", bronze_delta_tables)